In [7]:
import os

from langchain_ollama import OllamaLLM

from langchain_core.prompts import ChatPromptTemplate

import json

In [8]:
# Initialize the Ollama LLM with mistral model (CUDA-enabled)

llm = OllamaLLM(
    model="mistral"
)

In [ ]:
# Define the system prompt for UK financial planning instructor

system_template = """You are a UK financial planning instructor specializing in helping newcomers build credit. 

Provide advice in strictly valid JSON format with this exact structure:

{{
  "steps": ["step 1", "step 2", ...],
  "warnings": ["warning 1", "warning 2", ...],
  "timeframe_months": "X"
}}

Rules:
- steps: Array of 5-7 actionable steps for building UK credit
- warnings: Array of 3-5 important warnings or pitfalls to avoid
- timeframe_months: String representing realistic timeframe (e.g., "6-12")
- All advice must be UK-specific (electoral roll, UK banks, UK credit system)
- Consider the user's income level and lack of UK credit history
- Output ONLY the JSON, no additional text"""

# User profile context template

human_template = """User Profile:
- Income: £{income}/month
- No credit history in the UK
- Just moved to the UK
- Needs to build credit from scratch
- I plan to apply for 3 credit cards immediately. What impact will this have?

Provide your credit building plan:"""

# Create ChatPromptTemplate with system and human messages

prompt = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("human", human_template)
])

# Format the prompt with user data

messages = prompt.format_messages(income="2500")

In [ ]:
# Get response from LLM

response = llm.invoke(messages)

# Parse and validate JSON

try:
    credit_plan = json.loads(response)
    
    # Validate required keys
    required_keys = ["steps", "warnings", "timeframe_months"]
    for key in required_keys:
        if key not in credit_plan:
            raise ValueError(f"Missing required key: {key}")
    
    # Pretty print the validated JSON
    print("UK Credit Building Plan:")
    print(json.dumps(credit_plan, indent=2))
    
except json.JSONDecodeError as e:
    print(f"Error: Invalid JSON response from LLM")
    print(f"Raw response: {response}")
except ValueError as e:
    print(f"Validation error: {e}")
    print(f"Response: {response}")

In [9]:
# Import libraries for RAG pipeline

import glob

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_chroma import Chroma

In [10]:
# Load all markdown files from data folder

markdown_files = glob.glob("data/*.md")

documents = []

for file_path in markdown_files:
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
        documents.append({"content": content, "source": file_path})

print(f"Loaded {len(documents)} markdown files from data folder")

Loaded 16 markdown files from data folder


In [11]:
# Split documents into chunks using RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

chunks = []
for doc in documents:
    doc_chunks = text_splitter.create_documents(
        texts=[doc["content"]],
        metadatas=[{"source": doc["source"]}]
    )
    chunks.extend(doc_chunks)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")

Created 164 chunks from 16 documents


In [12]:
# Create embeddings using sentence-transformers and store in Chroma

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Initialize Chroma vector store

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

In [ ]:
# Print statistics

print(f"Total chunks created: {len(chunks)}")
print(f"Vector store persisted to: ./chroma_db")
print(f"Embedding model: all-MiniLM-L6-v2")

In [ ]:
# Load existing Chroma vector database and retrieve documents using MMR

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize embeddings (must match the embedding model used during storage)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Load the existing Chroma vector store
vector_store = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

# Define the query
query = "How can I build credit after moving to the UK?"

# Create retriever with MMR search type and k=5
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5}
)

# Retrieve documents using MMR
results = retriever.invoke(query)

# Print the retrieved chunks
print(f"\nQuery: {query}\n")
print(f"Retrieved {len(results)} documents using MMR search:\n")
print("=" * 80)

for i, doc in enumerate(results, 1):
    print(f"\n--- Chunk {i} ---")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")
    print(f"Content:\n{doc.page_content}")
    print("=" * 80)

In [15]:
# RAG Pipeline: Retrieve context and inject into prompt for Ollama

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
import json

# Initialize embeddings and load existing Chroma vector store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

# Create MMR retriever with k=5
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5}
)

# User query
user_query = "Explain how hard searches affect UK credit score"

# Retrieve relevant documents using MMR
retrieved_docs = retriever.invoke(user_query)

# Format retrieved context into a string
context = "\n\n".join([
    f"Source: {doc.metadata.get('source', 'Unknown')}\nContent: {doc.page_content}"
    for doc in retrieved_docs
])

# RAG-enabled system template with context injection
rag_system_template = """You are a UK financial planning instructor specializing in helping newcomers build credit. 

Use the following retrieved context to inform your advice:
{context}

Provide advice in strictly valid JSON format with this exact structure:

{{
  "steps": ["step 1", "step 2", ...],
  "warnings": ["warning 1", "warning 2", ...],
  "timeframe_months": "X"
}}

Rules:
- steps: Array of 5-7 actionable steps for building UK credit
- warnings: Array of 3-5 important warnings or pitfalls to avoid
- timeframe_months: String representing realistic timeframe (e.g., "6-12")
- All advice must be UK-specific (electoral roll, UK banks, UK credit system)
- Consider the user's income level and lack of UK credit history
- Use the retrieved context to provide accurate, evidence-based advice
- Output ONLY the JSON, no additional text"""

# RAG-enabled human template with user query
rag_human_template = """User Profile:
- Income: £{income}/month
- No credit history in the UK
- Just moved to the UK
- Needs to build credit from scratch

User Question: {query}

Provide your credit building plan based on the retrieved context:"""

# Create RAG ChatPromptTemplate
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", rag_system_template),
    ("human", rag_human_template)
])

# Initialize Ollama LLM
llm = OllamaLLM(model="gemma:2b-instruct")

# Format the RAG prompt with context, income, and query
rag_messages = rag_prompt.format_messages(
    context=context,
    income="2500",
    query=user_query
)

# Get response from LLM with retrieved context
rag_response = llm.invoke(rag_messages)

# Parse and validate JSON
try:
    credit_plan = json.loads(rag_response)
    
    # Validate required keys
    required_keys = ["steps", "warnings", "timeframe_months"]
    for key in required_keys:
        if key not in credit_plan:
            raise ValueError(f"Missing required key: {key}")
    
    # Pretty print the validated JSON
    print("UK Credit Building Plan (RAG-Enhanced):")
    print(json.dumps(credit_plan, indent=2))
    
except json.JSONDecodeError as e:
    print(f"Error: Invalid JSON response from LLM")
    print(f"Raw response: {rag_response}")
except ValueError as e:
    print(f"Validation error: {e}")
    print(f"Response: {rag_response}")

UK Credit Building Plan (RAG-Enhanced):
{
  "steps": [
    "1. Contact the lenders about your missed payments and discuss a payment arrangement or settlement.",
    "2. Make timely payments according to the agreed arrangement.",
    "3. Request a Notice of Correction (NoC) to explain the circumstances surrounding each missed payment.",
    "4. Pay any late payment fees or accumulated interest.",
    "5. Check your credit report regularly to ensure all payments are being reported correctly.",
    "6. Register on the UK Electoral Roll to establish your UK residency and help build your credit history.",
    "7. Consider applying for a 'credit builder' loan or a secured credit card to start building a positive credit history"
  ],
  "warnings": [
    "Missing payments can negatively impact your credit score, making it difficult to secure loans or services in the future.",
    "Do not ignore missed payments as they may result in County Court Judgments and further damage to your credit stand